---

# RA2 - Streaming Data Processing and Time Windows

Utilize os dados e o contexto do projeto para demonstrar o processamento em streaming e a aplicação de janelas temporais. Escolha um dos cenários:

- **Cenário A - Dados estáticos:** simule a chegada de novos registros utilizando dados artificiais coerentes com o contexto do projeto.
- **Cenário B - Dados em streaming:** utilize sua própria fonte de dados contínuos, sem necessidade de simulação.

## 8. Objetivo da análise em streaming

Responda brevemente:

- O que o grupo deseja monitorar ou analisar?
- Qual informação espera obter com o processamento em streaming?

**Resposta do grupo:**

Queremos acompanhar a **taxa de atraso de entrega** (pedidos entregues depois da data estimada) a cada **10 minutos**, conforme novos pedidos chegam. Isso se liga às perguntas 1 e 2 do RA1, sobre atrasos de entrega.

## 9. Fonte de dados e processamento

- Informe o cenário escolhido: dados reais em streaming ou simulação.
- Explique brevemente como os novos registros chegam.
- Implemente o processamento em tempo real.
- Demonstre o recebimento de novos registros.

**Resposta do grupo:**

**Cenário A (simulação):** a base da Olist é fixa, então criamos **15 pedidos artificiais**, escritos à mão, no formato da tabela `orders` (pedido, estado, data da entrega e data estimada).

- **Chegada:** os pedidos são divididos em **3 lotes** e gravados em arquivos CSV na pasta `data/stream_input/`, um de cada vez, com uma pausa entre eles.
- **Processamento:** o Spark vigia a pasta e processa cada arquivo novo. Para cada pedido, calcula `dias_atraso` (entrega menos estimada) e `atrasado` (1 se `dias_atraso` for maior que 0).
- **Ambiente:** rodamos em um container Linux (`docker/Dockerfile`), porque o Spark no Windows exige o `winutils`.

In [1]:
import shutil
import time
import warnings
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

warnings.filterwarnings("ignore")  # esconde um aviso do PySpark sobre a versao do pandas

spark = (
    SparkSession.builder.master("local[2]")
    .appName("ra2-streaming")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/21 00:41:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0


**Pedidos simulados:** 3 lotes de 5 pedidos, entregues no dia 01/05/2018 entre 12:00 e 12:35.

In [2]:
colunas = ["order_id", "customer_state", "entrega", "estimada"]

# Cada pedido: (id, estado, data e hora da entrega, data estimada de entrega)
lote1 = [
    ("p01", "SP", "2018-05-01 12:01:10", "2018-05-10"),
    ("p02", "RJ", "2018-05-01 12:03:45", "2018-04-29"),
    ("p03", "MG", "2018-05-01 12:05:20", "2018-05-08"),
    ("p04", "SP", "2018-05-01 12:07:05", "2018-05-12"),
    ("p05", "BA", "2018-05-01 12:08:30", "2018-04-30"),
]
lote2 = [
    ("p06", "SP", "2018-05-01 12:11:15", "2018-05-09"),
    ("p07", "RS", "2018-05-01 12:14:40", "2018-04-27"),
    ("p08", "PR", "2018-05-01 12:17:10", "2018-05-06"),
    ("p09", "SP", "2018-05-01 12:21:25", "2018-05-03"),
    ("p10", "MG", "2018-05-01 12:23:50", "2018-04-30"),
]
# o pedido p15 foi entregue as 12:12, mas só chega no lote 3 (chegada fora de ordem)
lote3 = [
    ("p11", "SC", "2018-05-01 12:26:30", "2018-05-05"),
    ("p12", "SP", "2018-05-01 12:28:10", "2018-04-29"),
    ("p13", "RJ", "2018-05-01 12:31:45", "2018-05-11"),
    ("p14", "SP", "2018-05-01 12:33:20", "2018-05-02"),
    ("p15", "BA", "2018-05-01 12:12:30", "2018-04-28"),
]

pd.DataFrame(lote1, columns=colunas)

,order_id,customer_state,entrega,estimada
0,p01,SP,2018-05-01 12:01:10,2018-05-10
1,p02,RJ,2018-05-01 12:03:45,2018-04-29
2,p03,MG,2018-05-01 12:05:20,2018-05-08
3,p04,SP,2018-05-01 12:07:05,2018-05-12
4,p05,BA,2018-05-01 12:08:30,2018-04-30


**Definição do stream:** o Spark lê a pasta de entrada e calcula, para cada pedido, `dias_atraso` e `atrasado`.

In [3]:
pasta = Path("../data/stream_input")
if pasta.exists():
    shutil.rmtree(pasta)
pasta.mkdir(parents=True)

esquema = "order_id STRING, customer_state STRING, entrega TIMESTAMP, estimada DATE"

entrada = spark.readStream.schema(esquema).option("header", True).csv(str(pasta))

pedidos = (
    entrada
    .withColumn("dias_atraso", F.datediff(F.to_date("entrega"), F.col("estimada")))
    .withColumn("atrasado", F.when(F.col("dias_atraso") > 0, 1).otherwise(0))
)

## 10. Aplicação de janelas temporais

- Utilize uma janela **Tumbling** (intervalos sem sobreposição) ou **Sliding** (intervalos que podem se sobrepor).
- Informe o tipo e a duração da janela e o deslocamento, quando aplicável.
- Implemente uma agregação coerente com a análise, como contagem, soma ou média.
- Explique brevemente o que será calculado em cada intervalo.

O foco é demonstrar a compreensão do conceito; não é necessário justificar tecnicamente a duração escolhida.

**Resposta do grupo:**

| Item | Valor |
|---|---|
| Tipo | **Tumbling** (janelas fixas, sem sobreposição) |
| Duração | **10 minutos** |
| Deslocamento | Não se aplica (em tumbling é igual à duração) |
| Data e hora usadas | `entrega` (momento da entrega) |
| Agregações | `count` (pedidos), `sum` (atrasados) e `avg` (taxa de atraso, em %) |
| Saída | `complete`: a tabela inteira é mostrada a cada lote |

**Em cada intervalo** calculamos: quantos pedidos foram entregues, quantos atrasaram e a taxa de atraso (%).

In [4]:
janelas = (
    pedidos
    .groupBy(F.window("entrega", "10 minutes"))
    .agg(
        F.count("*").alias("pedidos"),
        F.sum("atrasado").alias("atrasados"),
        F.round(F.avg("atrasado") * 100, 1).alias("taxa_atraso_pct"),
    )
)

consulta = (
    janelas.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("janelas")
    .start()
)
print("stream ativo:", consulta.isActive)

stream ativo: True


## 11. Resultados e interpretação

Apresente os resultados do processamento e demonstre como a análise se comporta com a chegada de novos registros.

**Chegada dos lotes:** cada chamada grava um arquivo novo na pasta e mostra a tabela das janelas.

In [5]:
def enviar_lote(numero, registros):
    """Grava um arquivo novo na pasta e espera o Spark processar."""
    pd.DataFrame(registros, columns=colunas).to_csv(pasta / f"lote_{numero}.csv", index=False)
    consulta.processAllAvailable()
    print(f"Lote {numero} chegou ({len(registros)} pedidos). Janelas agora:")
    spark.sql("""
        SELECT date_format(window.start, 'HH:mm') AS inicio,
               date_format(window.end, 'HH:mm') AS fim,
               pedidos, atrasados, taxa_atraso_pct
        FROM janelas
        ORDER BY window.start
    """).show()


enviar_lote(1, lote1)

Lote 1 chegou (5 pedidos). Janelas agora:


+------+-----+-------+---------+---------------+
|inicio|  fim|pedidos|atrasados|taxa_atraso_pct|
+------+-----+-------+---------+---------------+
| 12:00|12:10|      5|        2|           40.0|
+------+-----+-------+---------+---------------+



In [6]:
time.sleep(2)  # intervalo entre as chegadas
enviar_lote(2, lote2)

Lote 2 chegou (5 pedidos). Janelas agora:
+------+-----+-------+---------+---------------+
|inicio|  fim|pedidos|atrasados|taxa_atraso_pct|
+------+-----+-------+---------+---------------+
| 12:00|12:10|      5|        2|           40.0|
| 12:10|12:20|      3|        1|           33.3|
| 12:20|12:30|      2|        1|           50.0|
+------+-----+-------+---------+---------------+



In [7]:
time.sleep(2)
enviar_lote(3, lote3)

Lote 3 chegou (5 pedidos). Janelas agora:
+------+-----+-------+---------+---------------+
|inicio|  fim|pedidos|atrasados|taxa_atraso_pct|
+------+-----+-------+---------+---------------+
| 12:00|12:10|      5|        2|           40.0|
| 12:10|12:20|      4|        2|           50.0|
| 12:20|12:30|      4|        2|           50.0|
| 12:30|12:40|      2|        0|            0.0|
+------+-----+-------+---------+---------------+



**Conferência:** a soma dos pedidos nas janelas deve ser igual ao total de pedidos enviados.

In [8]:
total = spark.sql("SELECT SUM(pedidos) FROM janelas").collect()[0][0]
print("Total de pedidos nas janelas:", total, "(de", len(lote1) + len(lote2) + len(lote3), "enviados)")

consulta.stop()
spark.stop()
shutil.rmtree(pasta, ignore_errors=True)

Total de pedidos nas janelas: 15 (de 15 enviados)


**Resposta do grupo:**

- **Lote 1:** cria a janela `12:00-12:10` com 5 pedidos, 2 atrasados (40,0%).
- **Lote 2:** cria `12:10-12:20` (3 pedidos, 1 atrasado, 33,3%) e `12:20-12:30` (2 pedidos, 1 atrasado, 50,0%). A janela `12:00-12:10` não muda.
- **Lote 3:** cria `12:30-12:40` (2 pedidos, nenhum atrasado) e atualiza `12:20-12:30`, que passa a ter 4 pedidos, 2 atrasados (50,0%). O pedido das 12:12 só chega neste lote e atualiza a janela `12:10-12:20`, que passa a ter 4 pedidos, 2 atrasados (50,0%).
- A soma das janelas dá 15 pedidos, igual ao total enviado.

Os dados são simulados, então as taxas não representam a Olist. O que a demonstração mostra é o comportamento do processamento.

## 12. Considerações do RA2

Responda brevemente:

- O que foi possível observar com o processamento em streaming?
- Qual foi a contribuição das janelas temporais para a análise?
- Essa abordagem faria sentido em uma aplicação real do projeto? Por quê?

Na última resposta, o grupo pode reconhecer que o streaming foi utilizado apenas como exercício didático, se esse for o caso.

**Resposta do grupo:**

- **Observado:** a taxa de atraso por janela muda a cada chegada. Janelas novas aparecem, e janelas antigas são atualizadas quando chega um pedido que pertence a elas.
- **Contribuição das janelas:** dividem o fluxo contínuo em intervalos de 10 minutos que podem ser comparados.
- **Faria sentido na prática?** Em parte. Com entregas registradas continuamente, acompanhar o atraso por janela ajudaria a reagir a problemas logísticos. Aqui o streaming foi um **exercício didático** com dados simulados. Os dados da Olist são um histórico fechado (cerca de 130 pedidos por dia, em média), e janelas de horas ou de um dia seriam mais adequadas.